In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 500)
import numpy as np
import matplotlib.pyplot as plt

import requests
from bs4 import BeautifulSoup
import os
import s3fs
import ast
import json


from sklearn.linear_model import RidgeCV, ElasticNetCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_squared_error,mean_absolute_error,mean_absolute_percentage_error,r2_score

In [ ]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_OUT_S3 = '/data_movies_final_selected_merged.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

with fs.open(FILE_OUT_PATH, mode='rb') as f_out:
    data_movies_final_selected_merged = pd.read_csv(f_out)

In [ ]:
df = data_movies_final_selected_merged.sort_values(by='release_date').reset_index(drop=True)
weights=df['vote_count']
df=df.drop(['release_date', 'vote_count'], axis=1)

features = df.drop('vote_average', axis = 1)
label = df['vote_average']
features.dtypes

In [ ]:
data_movies_final_selected_merged=data_movies_final_selected_merged[data_movies_final_selected_merged['vote_count']>=5]

In [ ]:
num_features = features.select_dtypes(include= ['int32', 'float64', 'int64']).columns
num_features

In [ ]:
transformer = Pipeline(steps= [
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer(transformers= [
    ('features', transformer, num_features)
])

preprocessor.fit_transform(features)

In [ ]:
split_index = int(len(df) * 0.9)
df_train = df.iloc[:split_index]
df_val = df.iloc[split_index:]


# Vérification
print(f"Taille totale : {len(df)}")
print(f"Train : {len(df_train)} lignes ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test : {len(df_val)} lignes ({len(df_val)/len(df)*100:.1f}%)")

In [ ]:
X_train = df_train.drop(columns=['vote_average'])
y_train=df_train['vote_average']

X_val = df_val.drop(columns=['vote_average'])
y_val=df_val['vote_average']

In [ ]:
rf = RandomForestRegressor(random_state=42)

In [ ]:
pip_reg = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', rf)
])

In [ ]:
pip_reg_fitted = pip_reg.fit(X_train, y_train)
y_predict = pip_reg_fitted.predict(X_val)

In [ ]:
r2_score(y_val, y_predict)

In [ ]:
importances = rf.feature_importances_

# Créer un DataFrame pour mieux visualiser
feature_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.barh(feature_importances['Feature'], feature_importances['Importance'])
plt.gca().invert_yaxis()  # Pour avoir la plus importante en haut
plt.xlabel("Importance")
plt.title("Importance des variables (Random Forest)")
plt.show()


In [ ]:
import statsmodels.api as sm
import pandas as pd

def backward_selection_wls(X, y, weights=None, significance_level=0.05, verbose=True):
    """
    Sélection backward avec WLS (Weighted Least Squares)

    Paramètres
    ----------
    X : pd.DataFrame
        Variables explicatives.
    y : pd.Series ou np.array
        Variable dépendante.
    weights : array-like, optionnel
        Poids à utiliser pour WLS. Si None, OLS est utilisée.
    significance_level : float
        Seuil de p-value pour garder une variable.
    verbose : bool
        Si True, affiche les étapes du processus.

    Retour
    -------
    model : statsmodels.regression.linear_model.RegressionResultsWrapper
        Modèle WLS final ajusté.
    selected_vars : list
        Liste des variables sélectionnées.
    """

    X_ = X.copy()
    X_ = sm.add_constant(X_)  # Ajouter une constante pour l'interception
    selected_vars = list(X_.columns)

    while True:
        # Ajustement du modèle WLS
        model = sm.WLS(y, X_[selected_vars], weights=weights).fit()
        pvalues = model.pvalues.drop('const', errors='ignore')

        # Trouver la variable la moins significative
        worst_pval = pvalues.max()
        worst_var = pvalues.idxmax()

        if worst_pval > significance_level:
            if verbose:
                print(f"Suppression de '{worst_var}' (p = {worst_pval:.4f})")
            selected_vars.remove(worst_var)
        else:
            break

        # Arrêt si on n’a plus de variable à supprimer
        if len(selected_vars) <= 1:
            break

    if verbose:
        print("\nVariables finales :", selected_vars)

    final_model = sm.WLS(y, X_[selected_vars], weights=weights).fit()
    return final_model, selected_vars


In [ ]:
final_model = sm.WLS(y_train, sm.add_constant(X_train[selected_vars])).fit()